In [ ]:
from scipy import stats
import pandas as pd
import numpy as np

!gdown --id '17BhmQ08NEtvn7WwPp-OcZwz-2u-cceZm'  --output data.csv
data = pd.read_csv('/content/data.csv', encoding = 'latin1')
data.head()

/usr/local/lib/python3.10/dist-packages/gdown/cli.py:121: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=17BhmQ08NEtvn7WwPp-OcZwz-2u-cceZm
To: /content/data.csv
100% 11.4k/11.4k [00:00<00:00, 23.5MB/s]


,Glucose,BloodPressure,Insulin,Age,Outcome
0,89,66,94,21,0
1,137,40,168,33,2
2,78,50,88,26,0
3,197,70,543,53,2
4,189,60,846,59,2


In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from xgboost import plot_importance
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

In [ ]:
X = data.drop('Outcome', axis=1)
Y = data['Outcome']
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, shuffle = True, test_size = 0.3, random_state = 87)


In [ ]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Y_train shape:", Y_train.shape)
print("Y_test shape:", Y_test.shape)

X_train shape: (509, 4)
X_test shape: (219, 4)
Y_train shape: (509,)
Y_test shape: (219,)


In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

model = XGBClassifier(max_depth=10, learning_rate=0.1, n_estimators=1000,
                      reg_alpha=0.005, subsample=0.7,
                      gamma=0, objective='binary:logistic')

scaler = StandardScaler()
columns = X_train.columns
indexs_train = X_train.index
X_train = pd.DataFrame(scaler.fit_transform(X_train), index = indexs_train, columns = columns)
indexs_test = X_test.index
X_test = pd.DataFrame(scaler.fit_transform(X_test), index = indexs_test, columns = columns)

model.fit(X_train, Y_train)


score = model.score(X_train, Y_train) * 100
print("Training accuracy: ", score, "%")


scores = cross_val_score(model, X_train, Y_train, cv=5, scoring='accuracy') * 100
print("Mean Cross-Validation accuracy: ", scores.mean(), "%")


kfold = KFold(n_splits=10, shuffle=True)
kf_cv_scores = cross_val_score(model, X_train, Y_train, cv=kfold, scoring='accuracy') * 100
print("K-fold CV average accuracy: ", kf_cv_scores.mean(), "%")


Y_pred = model.predict(X_test)
accuracy = accuracy_score(Y_test, Y_pred)
print("Accuracy on test set: ", accuracy)


conf_matrix = confusion_matrix(Y_test, Y_pred)
print("Confusion Matrix:\n", conf_matrix)

class_report = classification_report(Y_test, Y_pred, digits = 8)

print("Classification Report:\n", class_report)


Training accuracy:  100.0 %
Mean Cross-Validation accuracy:  100.0 %
K-fold CV average accuracy:  100.0 %
Accuracy on test set:  0.9726027397260274
Confusion Matrix:
 [[53  0  1]
 [ 0 74  5]
 [ 0  0 86]]
Classification Report:
               precision    recall  f1-score   support

           0  1.00000000 0.98148148 0.99065421        54
           1  1.00000000 0.93670886 0.96732026        79
           2  0.93478261 1.00000000 0.96629213        86

    accuracy                      0.97260274       219
   macro avg  0.97826087 0.97273011 0.97475553       219
weighted avg  0.97438952 0.97260274 0.97267010       219




### LMCH Dataset

In [ ]:
!gdown --id '1-0NT8HBlFAkB_lMSxE8C0fMiy5V__0Zt'  --output data.csv
data = pd.read_csv('/content/data.csv', encoding = 'latin1')
data.head()


/usr/local/lib/python3.10/dist-packages/gdown/cli.py:121: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1-0NT8HBlFAkB_lMSxE8C0fMiy5V__0Zt
To: /content/data.csv
100% 25.4k/25.4k [00:00<00:00, 60.6MB/s]


,AGE,HbA1c,Chol,TG,HDL,VLDL,Output
0,50,4.9,4.2,0.9,2.4,0.5,0
1,26,4.9,3.7,1.4,1.1,0.6,0
2,50,4.9,4.2,0.9,2.4,0.5,0
3,50,4.9,4.2,0.9,2.4,0.5,0
4,33,4.9,4.9,1.0,0.8,0.4,0


In [ ]:
from sklearn.preprocessing import LabelEncoder
import keras
from keras.utils import to_categorical
#Label Encoding
labels = data['Output']
# encode class values as integers
encoder = LabelEncoder()
encoder.fit(labels)
encoded_Y = encoder.transform(labels)
# convert integers to dummy variables (i.e. one hot encoded)
dummy_y = to_categorical(encoded_Y)
# Remove the labels from the features
# axis 1 refers to the columns
data = data.drop('Output', axis = 1)

# Saving feature names for later use
data_list = list(data.columns)


In [ ]:

X_train, X_test, Y_train, Y_test = train_test_split(data, dummy_y, shuffle = True, test_size = 0.3, random_state = 87)


In [ ]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Y_train shape:", Y_train.shape)
print("Y_test shape:", Y_test.shape)

X_train shape: (699, 6)
X_test shape: (300, 6)
Y_train shape: (699, 5)
Y_test shape: (300, 5)


In [ ]:
model = XGBClassifier(max_depth=10, learning_rate=0.1, n_estimators=1000,
                      reg_alpha=0.005, subsample=0.8,
                      gamma=0, objective='binary:logistic')

scaler = StandardScaler()
columns = X_train.columns
indexs_train = X_train.index
X_train = pd.DataFrame(scaler.fit_transform(X_train), index = indexs_train, columns = columns)
indexs_test = X_test.index
X_test = pd.DataFrame(scaler.fit_transform(X_test), index = indexs_test, columns = columns)


model.fit(X_train, Y_train)


score = model.score(X_train, Y_train) * 100
print("Training accuracy: ", score, "%")


scores = cross_val_score(model, X_train, Y_train, cv=5, scoring='accuracy') * 100
print("Mean Cross-Validation accuracy: ", scores.mean(), "%")


kfold = KFold(n_splits=10, shuffle=True)
kf_cv_scores = cross_val_score(model, X_train, Y_train, cv=kfold, scoring='accuracy') * 100
print("K-fold CV average accuracy: ", kf_cv_scores.mean(), "%")


Y_pred = model.predict(X_test)
accuracy = accuracy_score(Y_test, Y_pred)
print("Accuracy on test set: ", accuracy)


#conf_matrix = confusion_matrix(Y_test, Y_pred)
#print("Confusion Matrix:\n", conf_matrix)

class_report = classification_report(Y_test, Y_pred, digits = 8)
print("Classification Report:\n", class_report)


Training accuracy:  99.71387696709584 %
Mean Cross-Validation accuracy:  96.71223021582735 %
K-fold CV average accuracy:  96.99171842650102 %
Accuracy on test set:  0.9533333333333334
Classification Report:
               precision    recall  f1-score   support

           0  0.89655172 0.81250000 0.85245902        32
           1  1.00000000 1.00000000 1.00000000        16
           2  0.96837945 0.98393574 0.97609562       249
           3  0.00000000 0.00000000 0.00000000         1
           4  0.00000000 0.00000000 0.00000000         2

   micro avg  0.96308725 0.95666667 0.95986622       300
   macro avg  0.57298623 0.55928715 0.56571093       300
weighted avg  0.95272046 0.95666667 0.95442166       300
 samples avg  0.95500000 0.95666667 0.95555556       300



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
